# 04 — Evaluation, confidence–coverage and figures

This is the **only** notebook that reads the sealed query labels. It is run
after both prediction files are frozen and parameters are never revisited.

In [ ]:
import sys
from pathlib import Path
REPO = Path.cwd()
if (REPO / "src").exists():
    sys.path.insert(0, str(REPO / "src"))
import heartmap
print("heartmap from:", Path(heartmap.__file__).parent)


In [ ]:
import pandas as pd
from heartmap.config import load_config
cfg = load_config("configs/main.yaml")
sealed = pd.read_csv(cfg.sealed_labels_path)
print("sealed labels opened:", len(sealed), "cells")
sealed.head(2)


## 1. Join predictions with truth and compute metrics

`evaluate_method` returns overall metrics (accuracy, balanced accuracy, macro-/weighted-F1), a per-class table, and confusion matrices, on both scopes:
- `all` (every query cell; out-of-reference types flagged)
- `closed_set` (true types present in the reference label set).

In [ ]:
import json
from heartmap.baseline import METHOD_NAME as BASE
from heartmap.models import METHOD_NAME as SCAN
from heartmap.metrics import evaluate_method
from heartmap.provenance import read_json

manifest = read_json(cfg.split_manifest_path)
ref_labels = manifest["reference_label_set"]
rows = {}
for method, stem in [(BASE, "baseline_predictions"),
                     (SCAN, "scanvi_predictions")]:
    p = pd.read_csv(cfg.results_dir / "predictions" /
                    f"{stem}_{cfg.run_tag}.csv")
    assert "true_cell_type" not in p.columns
    rows[method] = evaluate_method(
        p, sealed, ref_labels, method,
        min_support=cfg["evaluable_class_min_support"])

summary = pd.concat([rows[m].summary for m in rows], ignore_index=True)
summary


## 2. Per-class results

Rare types are never dropped. Mesothelial (n=9 in D6) is below the pre-registered evaluable threshold (20) but remains here and in the all-class macro-F1.

In [ ]:
per_class = pd.concat([rows[m].per_class for m in rows], ignore_index=True)
per_class[per_class.scope == "all"].pivot(
    index="label", columns="method",
    values=["precision", "recall", "f1", "support"])


## 3. Confusion matrices

In [ ]:
import matplotlib.pyplot as plt
for method in rows:
    cm = rows[method].confusion[f"{method}__closed_set__row_normalized"]
    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(cm.to_numpy(), vmin=0, vmax=1, cmap="Blues")
    ax.set_xticks(range(len(cm.columns)), cm.columns, rotation=90)
    ax.set_yticks(range(len(cm.index)), cm.index)
    ax.set_title(f"{method} (closed-set, row-normalised)")
    ax.set_xlabel("predicted"); ax.set_ylabel("true")
    fig.colorbar(im, ax=ax, fraction=0.046)
    plt.show()


## 4. Confidence–coverage sweep

At t=0 coverage is 1 by construction; coverage cannot increase with the threshold. If accuracy on retained cells rises, the correct statement is that high-confidence predictions were more reliable **for cells of this one held-out donor**.

In [ ]:
from heartmap.confidence import confidence_coverage
cov_parts = []
for method in rows:
    cov, rej = confidence_coverage(
        rows[method].joined, list(cfg["confidence_thresholds"]), ref_labels)
    cov.insert(0, "method", method)
    cov_parts.append(cov)
coverage = pd.concat(cov_parts, ignore_index=True)
coverage


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
for method, g in coverage.groupby("method"):
    g = g.sort_values("threshold")
    axes[0].plot(g.coverage, g.accuracy_on_retained, marker="o", label=method)
    axes[1].plot(g.coverage, g.macro_f1_on_retained, marker="o", label=method)
axes[0].set_xlabel("coverage"); axes[0].set_ylabel("accuracy on retained")
axes[1].set_xlabel("coverage"); axes[1].set_ylabel("macro-F1 on retained")
for ax in axes: ax.legend(); ax.invert_xaxis()
plt.tight_layout(); plt.show()


## 5. Generate the canonical figures

`make_figures.py` writes the joint-latent UMAPs (fitted on `X_scANVI`, fixed seed; visual only), confusion heatmaps, method comparison, F1 vs abundance and coverage plots, plus a figures manifest.

In [ ]:
# shell equivalent: python scripts/make_figures.py --config configs/main.yaml
import subprocess, sys
r = subprocess.run(
    [sys.executable, "scripts/make_figures.py", "--config",
     "configs/main.yaml"], capture_output=True, text=True)
print(r.stdout); print(r.stderr[-1000:] if r.returncode else "")
assert r.returncode == 0


## 6. Write `results/analysis_summary.md`

A short, hedged, auto-generated summary of the numbers. One held-out donor = one case study; cells are not donor replicates.

In [ ]:
oor = summary[summary.scope == "all"][
    ["method", "n_out_of_reference_types",
     "n_out_of_reference_cells"]].drop_duplicates()
lines = ["# Analysis summary (auto-generated)", "",
         f"- Query donor: {manifest['query_donor']} "
         f"({int(manifest['n_query_cells'])} cells); reference: "
         f"{len(manifest['reference_donor_ids'])} donors, "
         f"{int(manifest['n_reference_cells'])} cells.", ""]
for _, r_ in summary.iterrows():
    lines.append(
        f"- {r_.method} [{r_.scope}]: accuracy={r_.accuracy:.3f}, "
        f"balanced_accuracy={r_.balanced_accuracy:.3f}, "
        f"macro_f1={r_.macro_f1:.3f}, weighted_f1={r_.weighted_f1:.3f}")
lines += ["", "Confidence is a prediction confidence score, not a calibrated "
              "probability. Results describe one held-out donor only."]
out = cfg.results_dir / "analysis_summary.md"
out.write_text("\n".join(lines))
print("\n".join(lines))


## 7. Verify everything

`--require-complete-main-run` exits non-zero unless a genuine main run (both prediction sets, metrics, figures) is present.

In [ ]:
r = subprocess.run([sys.executable, "scripts/verify_outputs.py",
                    "--config", "configs/main.yaml",
                    "--require-complete-main-run"])
assert r.returncode == 0, "main run verification failed"
